In [3]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

# ──────────────────────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ──────────────────────────────────────────────────────────────────────────────
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

# ──────────────────────────────────────────────────────────────────────────────
# 2. TIME & CALENDAR FEATURES (The 92-Point Addition)
# ──────────────────────────────────────────────────────────────────────────────
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # Parse standard time
    parts = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour'] = parts[0]
    df['minute'] = parts[1]
    df['time_slot'] = df['hour'] * 4 + df['minute'] // 15
    
    # Cyclical time continuous waves
    df['sin_time'] = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['cos_time'] = np.cos(2 * np.pi * df['time_slot'] / 96)
    
    # Categorical Time (For CatBoost Target Encoding)
    df['time_slot_cat'] = 'Slot_' + df['time_slot'].astype(str)
    
    # 🚀 CALENDAR FEATURES: The Missing Link 🚀
    # Assuming day is a continuous counter, modulo 7 gives us the day of the week
    df['day_of_week'] = (df['day'] % 7).astype(str)
    
    # Assuming days 5 and 6 in the 7-day cycle represent the weekend
    df['is_weekend'] = df['day_of_week'].isin(['5', '6']).astype(str)
    
    return df

train = build_features(train)
test  = build_features(test)

# ──────────────────────────────────────────────────────────────────────────────
# 3. CLEANUP & CATEGORICALS
# ──────────────────────────────────────────────────────────────────────────────
train['Temperature'] = train['Temperature'].fillna(-999)
test['Temperature']  = test['Temperature'].fillna(-999)

CAT_FEATURES = [
    'geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 
    'time_slot_cat', 'day_of_week', 'is_weekend'
]

for col in CAT_FEATURES:
    train[col] = train[col].fillna('Unknown').astype(str)
    test[col]  = test[col].fillna('Unknown').astype(str)

# ──────────────────────────────────────────────────────────────────────────────
# 4. FEATURE SELECTION & VALIDATION SPLIT
# ──────────────────────────────────────────────────────────────────────────────
FEATURES = [
    'geohash', 'hour', 'minute', 'time_slot', 'time_slot_cat',
    'day_of_week', 'is_weekend',
    'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks',
    'Temperature', 'Weather', 'sin_time', 'cos_time'
]
TARGET = 'demand'

# Quick 85/15 split just to find the optimal stopping point
X_train, X_val, y_train, y_val = train_test_split(
    train[FEATURES], train[TARGET], test_size=0.15, random_state=42
)

# ──────────────────────────────────────────────────────────────────────────────
# 5. FIND OPTIMAL ITERATIONS
# ──────────────────────────────────────────────────────────────────────────────
MODEL_PARAMS = dict(
    learning_rate= 0.035,
    depth        = 8,       # Increased depth to allow geohash + day + time to combine
    l2_leaf_reg  = 5,
    loss_function= 'RMSE',
    eval_metric  = 'RMSE',
    od_type      = 'Iter',
    od_wait      = 250,
    verbose      = 500,
)

print("Finding optimal tree count on Validation Set...")
val_model = CatBoostRegressor(iterations=6000, **MODEL_PARAMS, random_seed=42)
val_model.fit(X_train, y_train, cat_features=CAT_FEATURES, eval_set=(X_val, y_val))

val_preds = np.clip(val_model.predict(X_val), 0, None)
print(f"\nValidation R²: {r2_score(y_val, val_preds):.5f}")

best_iters = val_model.best_iteration_
print(f"Optimal iterations found: {best_iters}")

# ──────────────────────────────────────────────────────────────────────────────
# 6. THE 100% DATA SEED ENSEMBLE
# ──────────────────────────────────────────────────────────────────────────────
print(f"\nTraining 5-Seed Ensemble on 100% of data ({len(train)} rows)...")
SEEDS = [42, 100, 777, 2024, 888]
test_preds = np.zeros(len(test))

for i, seed in enumerate(SEEDS):
    print(f"  --> Training Seed {i+1}/5 (Seed {seed}) ...")
    
    # Safely copy and update the parameters so we don't pass duplicates
    ensemble_params = MODEL_PARAMS.copy()
    ensemble_params['iterations']  = best_iters
    ensemble_params['random_seed'] = seed
    ensemble_params['verbose']     = 0   # Silences the output for the loop
    
    # Train directly on all available data without stopping early
    seed_model = CatBoostRegressor(**ensemble_params)
    seed_model.fit(train[FEATURES], train[TARGET], cat_features=CAT_FEATURES)
    
    # Add predictions to the blend
    test_preds += seed_model.predict(test[FEATURES]) / len(SEEDS)

test_preds = np.clip(test_preds, 0, None)
submission = pd.DataFrame({'Index': test['Index'], 'demand': test_preds})
submission.to_csv('submission_92_target.csv', index=False)
print("\nSuccess! Saved to submission_92_target.csv")

Finding optimal tree count on Validation Set...
0:	learn: 0.1384828	test: 0.1385522	best: 0.1385522 (0)	total: 71.4ms	remaining: 7m 8s
500:	learn: 0.0364717	test: 0.0366679	best: 0.0366679 (500)	total: 1m 15s	remaining: 13m 45s
1000:	learn: 0.0330248	test: 0.0344555	best: 0.0344555 (1000)	total: 2m 31s	remaining: 12m 36s
1500:	learn: 0.0307783	test: 0.0332235	best: 0.0332235 (1500)	total: 3m 52s	remaining: 11m 38s
2000:	learn: 0.0293093	test: 0.0325903	best: 0.0325903 (2000)	total: 5m 12s	remaining: 10m 25s
2500:	learn: 0.0281956	test: 0.0321986	best: 0.0321979 (2496)	total: 6m 33s	remaining: 9m 9s
3000:	learn: 0.0273304	test: 0.0319440	best: 0.0319440 (3000)	total: 7m 51s	remaining: 7m 51s
3500:	learn: 0.0265903	test: 0.0317678	best: 0.0317673 (3498)	total: 9m 10s	remaining: 6m 32s
4000:	learn: 0.0259493	test: 0.0316223	best: 0.0316223 (4000)	total: 10m 28s	remaining: 5m 14s
4500:	learn: 0.0253572	test: 0.0315135	best: 0.0315118 (4453)	total: 11m 49s	remaining: 3m 56s
5000:	learn: 0.0